# Task 2b: Local Window Sum With `torch.nn.RNN`

For each position `t`, the target is the sum of the input values inside a centered window of radius `K`.

## TODO

Implement `RNNModel` using only `torch.nn` modules.

- Use `nn.Embedding` for integer tokens.
- Use `nn.RNN(..., batch_first=True)` for sequence processing.
- Use `nn.Linear` to classify each time step into a possible window sum.
- The largest possible target is `9 * (2 * K + 1)`, so `output_vocab` must be `9 * (2 * K + 1) + 1`.
- Return logits shaped `(B, T, output_vocab)` with no `softmax`.

In [1]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = 'cuda' if torch.cuda.is_available() else 'cpu'
set_seed(217)
K = 2
device

'cpu'

In [2]:
class CustomDataset(Dataset):
    def __init__(self, vocab_size=10, seq_len=25, size=40000, K=2):
        self.vocab_size = vocab_size
        self.seq_len = seq_len
        self.size = size
        self.K = K
        self.X = torch.randint(0, vocab_size, (size, seq_len))
        x_floated = self.X.unsqueeze(1).float()
        kernel = torch.ones((1, 1, 2 * K + 1))
        self.Y = F.conv1d(x_floated, kernel, padding='same').squeeze(1).long()

    def __len__(self):
        return self.size

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

## Model TODO

The model needs context from nearby positions. A vanilla `nn.RNN` reads left-to-right, so start with a regular RNN and consider `bidirectional=True` if your accuracy is weak. If you make it bidirectional, remember the linear layer input size becomes `2 * d_model`.

In [5]:
class RNNModel(nn.Module):
    def __init__(self, K, d_model=64, num_layers=2):
        super().__init__()
        self.input_vocab = 10
        self.output_vocab = 9 * (2 * K + 1) + 1

        self.embedding = nn.Embedding(self.input_vocab, d_model)
        self.rnn = nn.RNN(d_model, d_model, num_layers=num_layers,
                          batch_first=True, bidirectional=True)
        self.fc = nn.Linear(2 * d_model, self.output_vocab)

    def forward(self, x):
        # x: (B, T)
        embedded = self.embedding(x)                 # (B, T, d_model)
        rnn_out, _ = self.rnn(embedded)              # (B, T, 2 * d_model)
        logits = self.fc(rnn_out)                    # (B, T, output_vocab)
        return logits

In [6]:
def evaluate(model, loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            preds = model(x).argmax(dim=-1)
            correct += (preds == y).sum().item()
            total += y.numel()
    return correct / total

def train(K=2, epochs=20, batch_size=256):
    model = RNNModel(K=K).to(device)
    dataset = CustomDataset(K=K)
    train_size = int(0.8 * len(dataset))
    test_size = len(dataset) - train_size
    train_set, test_set = random_split(dataset, [train_size, test_size])
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_set, batch_size=64)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits.view(-1, logits.size(-1)), y.view(-1))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f'Epoch {epoch + 1}: loss={total_loss / len(train_loader):.4f}, acc={evaluate(model, test_loader, device) * 100:.2f}%')
    return model

# Run after completing RNNModel:
model = train(K=K)

Epoch 1: loss=1.5315, acc=89.15%
Epoch 2: loss=0.6612, acc=85.63%
Epoch 3: loss=0.2267, acc=98.87%
Epoch 4: loss=0.0909, acc=99.30%
Epoch 5: loss=0.0527, acc=99.46%
Epoch 6: loss=0.0365, acc=99.45%
Epoch 7: loss=0.0277, acc=99.54%
Epoch 8: loss=0.0225, acc=99.67%
Epoch 9: loss=0.0181, acc=99.57%
Epoch 10: loss=0.0143, acc=99.82%
Epoch 11: loss=0.0119, acc=99.75%
Epoch 12: loss=3.1711, acc=23.08%
Epoch 13: loss=1.7731, acc=56.52%
Epoch 14: loss=1.3528, acc=56.71%
Epoch 15: loss=1.1667, acc=65.56%
Epoch 16: loss=1.0632, acc=56.26%
Epoch 17: loss=0.9792, acc=79.57%
Epoch 18: loss=0.9107, acc=73.83%
Epoch 19: loss=0.8307, acc=81.22%
Epoch 20: loss=0.7609, acc=89.18%
